# プロパティとアクセサ

## 概要

プロパティは、C#におけるカプセル化を実現する重要な機能です。クラスのフィールドに対する読み書きを制御し、データの整合性を保ちながら、使いやすいインターフェースを提供します。

## プロパティの基本

### アクセサの構文バリエーション

プロパティのアクセサには、以下のような構文バリエーションがあります：

1. ブロック構文（従来の方法）

In [ ]:
private string _name;
public string Name
{
    get { return _name; }
    set { _name = value; }
}

2. 式形式（C# 6.0以降）

In [ ]:
private string _name;
public string Name
{
    get => _name;
    set => _name = value;
}

3. 自動実装プロパティの省略形

In [ ]:
// コンパイラがバッキングフィールドを自動生成
public string Name { get; set; }

4. 計算プロパティの式形式

In [ ]:
// 読み取り専用の計算プロパティ
public string FullName => $"{FirstName} {LastName}";

### アクセサのアクセス修飾子

個々のアクセサに対して、プロパティ本体より制限の強いアクセス修飾子を設定できます：

In [ ]:
public class Person
{
    // プロパティは public だが、set は private
    public string Name { get; private set; }
    
    // プロパティは public だが、set は protected
    public int Age { get; protected set; }
    
    // バッキングフィールドを使用した場合
    private string _email;
    public string Email 
    {
        get => _email;
        protected set => _email = value?.ToLower();
    }
}

#### アクセス修飾子の使用例

In [ ]:
public class Document
{
    // 内部での設定のみ許可
    public string Title { get; private set; }
    
    // 派生クラスでの設定を許可
    public string Author { get; protected set; }
    
    // 読み取りは public、設定は派生クラスのみ
    public DateTime Created { get; protected set; }
    
    // 読み取りは public、設定は同一アセンブリのみ
    public DateTime LastModified { get; internal set; }
}

#### 継承時の注意点

以下のコードはエラーになります：

In [3]:
public class BaseClass
{
    // protected set により派生クラスからの設定を許可
    public virtual string Data { get; protected set; }
}

public class DerivedClass : BaseClass
{
    // オーバーライド時にアクセス修飾子を変更できない
    public override string Data 
    { 
        get => base.Data;
        // protected を public に緩和しようとするとエラー
        set => base.Data = value;
    }
}

Error: (14,9): error CS0507: 'DerivedClass.Data.set': 'protected' の継承メンバー 'BaseClass.Data.set' をオーバーライドするときに、アクセス修飾子を変更できません

正しくは：

In [4]:
public class BaseClass
{
    // protected set により派生クラスからの設定を許可
    public virtual string Data { get; protected set; }
}

public class DerivedClass : BaseClass
{
    // オーバーライド時は基底クラスと同じアクセス修飾子を使用する必要がある
    public override string Data 
    { 
        get => base.Data;
        protected set => base.Data = value;  // protected のまま
    }
}

// アクセス修飾子を変更したい場合は、新しいプロパティを定義する
public class AnotherDerivedClass : BaseClass
{
    // 基底クラスのプロパティはそのまま
    // 新しい public なプロパティを追加
    public string PublicData
    {
        get => Data;
        set => Data = value;  // protected set を通じて設定
    }
}

### 自動実装プロパティ

最もシンプルな形式のプロパティは、自動実装プロパティです。

In [ ]:
public class Person
{
    public string Name { get; set; }
    public int Age { get; set; }
}

これは以下の完全な実装と同等です：

In [1]:
public class Person
{
    private string _name;
    private int _age;
    
    public string Name
    {
        get { return _name; }
        set { _name = value; }
    }
    
    public int Age
    {
        get { return _age; }
        set { _age = value; }
    }
}

### 読み取り専用プロパティ

データの不変性を保証したい場合、set アクセサを省略できます：

In [ ]:
public class ImmutablePerson
{
    private readonly string _name;
    
    public ImmutablePerson(string name)
    {
        _name = name;
    }
    
    public string Name { get => _name; }
}

C# 9.0以降では、init アクセサを使用してオブジェクト初期化時のみ値を設定できます：

In [ ]:
public class ModernPerson
{
    public string Name { get; init; }
}

### 計算プロパティ

プロパティは単なるデータの格納だけでなく、計算結果を返すこともできます：

In [ ]:
public class Rectangle
{
    public double Width { get; set; }
    public double Height { get; set; }
    
    public double Area
    {
        get { return Width * Height; }
    }
}

### バリデーション付きプロパティ

プロパティは値の検証を行うことができます：

In [ ]:
public class Employee
{
    private int _age;
    
    public int Age
    {
        get { return _age; }
        set
        {
            if (value < 0 || value > 150)
            {
                throw new ArgumentException("年齢は0から150の間で設定してください。");
            }
            _age = value;
        }
    }
}

## プロパティの応用例

### 通知プロパティ

値が変更されたことを通知する必要がある場合：

In [2]:
using System.ComponentModel;

public class NotifyingPerson : INotifyPropertyChanged
{
    private string _name;
    
    public event PropertyChangedEventHandler PropertyChanged;
    
    public string Name
    {
        get { return _name; }
        set
        {
            if (_name != value)
            {
                _name = value;
                PropertyChanged?.Invoke(this, new PropertyChangedEventArgs(nameof(Name)));
            }
        }
    }
}

### 遅延評価プロパティ

計算コストの高い値を必要になるまで計算しない場合：

In [ ]:
public class ExpensiveCalculation
{
    private double? _cachedResult;
    
    public double Result
    {
        get
        {
            if (!_cachedResult.HasValue)
            {
                _cachedResult = PerformExpensiveCalculation();
            }
            return _cachedResult.Value;
        }
    }
    
    private double PerformExpensiveCalculation()
    {
        // 重い計算処理
        System.Threading.Thread.Sleep(1000); // シミュレーション
        return 42.0;
    }
}

#### Lazy<T>を使用した遅延評価

C#では `Lazy<T>` クラスを使用することで、より簡潔に遅延評価を実装できます：

In [ ]:
public class ModernExpensiveCalculation
{
    private readonly Lazy<double> _lazyResult;
    
    public ModernExpensiveCalculation()
    {
        _lazyResult = new Lazy<double>(() => PerformExpensiveCalculation());
        // スレッドセーフな実装が必要な場合は以下のようにもできます
        // _lazyResult = new Lazy<double>(() => PerformExpensiveCalculation(), 
        //     LazyThreadSafetyMode.ExecutionAndPublication);
    }
    
    public double Result
    {
        get { return _lazyResult.Value; }
    }
    
    private double PerformExpensiveCalculation()
    {
        // 重い計算処理
        System.Threading.Thread.Sleep(1000); // シミュレーション
        return 42.0;
    }
}

`Lazy<T>` を使用する利点：

* スレッドセーフな実装が容易
* 初期化ロジックをコンストラクタでまとめて定義可能
* 初期化状態の確認が容易（IsValueCreatedプロパティ）
* 初期化の例外handling

## 演習問題

### 問題１：基本的なバリデーション

以下の `BankAccount` クラスを完成させてください：

In [ ]:
public class BankAccount
{
    // TODO: 残高(Balance)プロパティを実装してください
    // 条件:
    // - 残高は0未満にならない
    // - 残高が変更されるたびに、取引履歴に記録する
    
    private List<string> _transactionHistory = new List<string>();
    public IReadOnlyList<string> TransactionHistory => _transactionHistory.AsReadOnly();
}

### 問題2：計算プロパティ

以下の `Circle` クラスを完成させてください：

In [ ]:
public class Temperature
{
    // TODO: 摂氏温度(Celsius)プロパティを実装してください
    // TODO: 華氏温度(Fahrenheit)プロパティを実装してください
    // TODO: ケルビン温度(Kelvin)プロパティを実装してください
    
    // 条件:
    // - いずれの単位で温度を設定しても、他の単位の温度が正しく計算される
    // - 絶対零度未満の温度は設定できない
    // - 温度が変更されるたびにログを出力する
}

## 解答例

各演習問題の解答例は別途提供しますが、まずは自力で実装にチャレンジしてください。解答を見る前に、以下の点を考えてみましょう：

1. プロパティでバリデーションをどのように実装するか
1. 読み取り専用プロパティと変更可能プロパティをどのように使い分けるか
1. プロパティ間の依存関係をどのように管理するか
1. 値の変更通知をどのように実装するか

## まとめ

プロパティとアクセサは、以下の目的で使用します：

* データのカプセル化
* 値の検証
* 計算値の提供
* 変更通知
* 遅延評価

適切なプロパティの設計は、クラスの使いやすさと保守性を大きく向上させます。